In [1]:
import json
import openai
import pandas as pd

from bertopic.representation import OpenAI
from bertopic import BERTopic

from sentence_transformers import SentenceTransformer

In [2]:
df = pd.read_csv('hotel_reviews_with_transl.csv', sep = '\t')
df.sample(3)

,id,hotel,review,lang,reviews_transl,reviews_len
2145,2272,Radisson,Our favorite hotel in London My family and I h...,en,Our favorite hotel in London My family and I h...,504
9059,9514,Millemiun,absolutely terrible!! My husband and I pre-boo...,en,absolutely terrible!! My husband and I pre-boo...,2126
3204,3372,Holiday Inn,Will definately return. Just returned after a ...,en,Will definately return. Just returned after a ...,732


In [3]:
docs = list(df.reviews_transl)

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired

representation_model = KeyBERTInspired()

vectorizer_model = CountVectorizer(min_df=5, stop_words = 'english')
topic_model = BERTopic(nr_topics = 'auto',
                       vectorizer_model = vectorizer_model,
                       representation_model = representation_model)
topics, ini_probs = topic_model.fit_transform(docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
topic_info = topic_model.get_topic_info()
topic_info.head()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,6600,-1_hotel_hotels_rooms_london,"[hotel, hotels, rooms, london, room, inn, bed,...",[Actually very good - but ask for a good room!...
1,0,532,0_hotels_hotel_london_rooms,"[hotels, hotel, london, rooms, facilities, pla...",[Oh stop moaning you're in London! I've used t...
2,1,372,1_hotel_luxurious_rooms_place,"[hotel, luxurious, rooms, place, nearby, resta...",[Good location very nice hotel. The hotels loc...
3,2,276,2_victoria_london_park_hotel,"[victoria, london, park, hotel, buckingham, ho...","[PARK PLAZA VICTORIA is fantastic, Park Plaza ..."
4,3,231,3_paddington_heathrow_london_hotel,"[paddington, heathrow, london, hotel, hotels, ...",[Great Location - Average Property A very larg...


In [6]:
topic_info.Representative_Docs

0     [Actually very good - but ask for a good room!...
1     [Oh stop moaning you're in London! I've used t...
2     [Good location very nice hotel. The hotels loc...
3     [PARK PLAZA VICTORIA is fantastic, Park Plaza ...
4     [Great Location - Average Property A very larg...
                            ...                        
89    [Excellent Hotel Great Staff, Great Service. E...
90    [Very convenient location for theatres On arri...
91    [Comfortable stay for convenience only Got a g...
92    [Good value can be noisy I stay here regularly...
93    [Good value good location, Good value good loc...
Name: Representative_Docs, Length: 94, dtype: object

In [7]:
# 方法1: トピックごとに代表ドキュメント1件のみ使用
repr_docs = [docs[0] for docs in topic_info.Representative_Docs]
print(f"代表ドキュメント数: {len(repr_docs)}")

代表ドキュメント数: 94


In [8]:
delimiter = '####'
system_message = "You're a helpful assistant. Your task is to analyse hotel reviews."
user_message = f'''Below is a representative set of customer reviews delimited with {delimiter}. 
Please, identify the main topics mentioned in these comments. 

Return a list of 10-20 topics. 
Output is a JSON list with the following format
[
    {{"topic_name": "<topic1>", "topic_description": "<topic_description1>"}}, 
    {{"topic_name": "<topic2>", "topic_description": "<topic_description2>"}},
    ...
]

Customer reviews:
{delimiter}
{delimiter.join(repr_docs)}
{delimiter}
/no_think'''

messages = [  
    {'role': 'system', 'content': system_message},    
    {'role': 'user', 'content': user_message},  
]

In [9]:
# プロンプトの入力トークン数を確認 (qwen3:1.7b のコンテキスト長は 40,960)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
n_tokens = len(tokenizer.encode(user_message))
print(f"入力トークン数: {n_tokens} / 40960")

入力トークン数: 27138 / 40960


In [10]:
import requests

# Structured Output: JSON Schema で出力形式を厳密に強制する
topic_schema = {
    "type": "object",
    "properties": {
        "topics": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "topic_name": {"type": "string"},
                    "topic_description": {"type": "string"}
                },
                "required": ["topic_name", "topic_description"]
            }
        }
    },
    "required": ["topics"]
}

resp = requests.post(
    'http://host.docker.internal:11434/v1/chat/completions',
    json={
        "model": "qwen3:1.7b",
        "messages": messages,
        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": "topics",
                "schema": topic_schema
            }
        },
    }
)
print(resp.status_code)

200


In [11]:
resp.json()

{'id': 'chatcmpl-295',
 'object': 'chat.completion',
 'created': 1776156009,
 'model': 'qwen3:1.7b',
 'system_fingerprint': 'fp_ollama',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': '{"topics": [{"topic_name": "Location", "topic_description": "All reviews highlight the hotels\' convenient locations in London, with some noting proximity to landmarks or city views. However, a few mention noise from nearby locations (e.g., busy streets, transport)."}, {"topic_name": "Room Facilities", "topic_description": "Rooms are described as clean, comfortable, and well-maintained. Some reviews note specific amenities like balcony views, modern furnishings, and good Wi-Fi. A few mention that certain services (e.g., spa, fitness) are available but may be limited or expensive."}, {"topic_name": "Service Quality", "topic_description": "Service is generally positive, with reviews praising staff for being helpful and attentive. However, some note that the concierge may not b

In [12]:
topic_response = json.loads(resp.json()['choices'][0]['message']['content'])
print(json.dumps(topic_response, indent=2, ensure_ascii=False))

{
  "topics": [
    {
      "topic_name": "Location",
      "topic_description": "All reviews highlight the hotels' convenient locations in London, with some noting proximity to landmarks or city views. However, a few mention noise from nearby locations (e.g., busy streets, transport)."
    },
    {
      "topic_name": "Room Facilities",
      "topic_description": "Rooms are described as clean, comfortable, and well-maintained. Some reviews note specific amenities like balcony views, modern furnishings, and good Wi-Fi. A few mention that certain services (e.g., spa, fitness) are available but may be limited or expensive."
    },
    {
      "topic_name": "Service Quality",
      "topic_description": "Service is generally positive, with reviews praising staff for being helpful and attentive. However, some note that the concierge may not be as responsive or knowledgeable, and a few mention that specific services (e.g., breakfast, concierge) can be costly."
    },
    {
      "topic_name"

In [13]:
### 方法2: 全代表ドキュメントからランダムサンプリング

import random

# 全トピックの代表ドキュメントをフラットにまとめる
all_repr_docs = topic_info.Representative_Docs.sum()
print(f"全代表ドキュメント数: {len(all_repr_docs)}")

# コンテキスト長に収まるようサンプリング
# プロンプトのテンプレート部分 (~200トークン) + 出力用の余裕 (~2000トークン) を差し引く
max_input_tokens = 40960 - 2200
random.seed(42)
random.shuffle(all_repr_docs)

sampled_docs = []
total_tokens = 0
for doc in all_repr_docs:
    doc_tokens = len(tokenizer.encode(doc))
    if total_tokens + doc_tokens > max_input_tokens:
        break
    sampled_docs.append(doc)
    total_tokens += doc_tokens

print(f"サンプリング後: {len(sampled_docs)} 件, {total_tokens} tokens")

全代表ドキュメント数: 282
サンプリング後: 122 件, 38577 tokens


In [14]:
delimiter = '####'
system_message = "You're a helpful assistant. Your task is to analyse hotel reviews."
user_message_sampled = f'''Below is a representative set of customer reviews delimited with {delimiter}. 
Please, identify the main topics mentioned in these comments. 

Return a list of 10-20 topics. 
Output is a JSON list with the following format
[
    {{"topic_name": "<topic1>", "topic_description": "<topic_description1>"}}, 
    {{"topic_name": "<topic2>", "topic_description": "<topic_description2>"}},
    ...
]

Customer reviews:
{delimiter}
{delimiter.join(sampled_docs)}
{delimiter}
/no_think'''

messages_sampled = [  
    {'role': 'system', 'content': system_message},    
    {'role': 'user', 'content': user_message_sampled},  
]

n_tokens = len(tokenizer.encode(user_message_sampled))
print(f"入力トークン数: {n_tokens} / 40960")

入力トークン数: 38798 / 40960


In [15]:
resp_sampled = requests.post(
    'http://host.docker.internal:11434/v1/chat/completions',
    json={
        "model": "qwen3:1.7b",
        "messages": messages_sampled,
        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": "topics",
                "schema": topic_schema
            }
        },
    }
)
print(resp_sampled.status_code)

200


In [16]:
resp_sampled.json()

{'id': 'chatcmpl-371',
 'object': 'chat.completion',
 'created': 1776156045,
 'model': 'qwen3:1.7b',
 'system_fingerprint': 'fp_ollama',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': '{\n  "topics": [\n    {\n      "topic_name": "Hotel Reviews in London",\n      "topic_description": "Analysis of reviews and complaints about various hotels in London, focusing on key issues and common themes."\n    },\n    {\n      "topic_name": "Positive Aspects",\n      "topic_description": "Key positives mentioned in the reviews: good locations, clean rooms, friendly staff, and value for money."\n    },\n    {\n      "topic_name": "Negative Aspects",\n      "topic_description": "Common complaints: lack of air conditioning, rude staff, high prices, poor room conditions, and issues with check-in procedures."\n    },\n    {\n      "topic_name": "Hotels Mentioned",\n      "topic_description": "Examples of hotels reviewed: The London Farringdon (affordable, good location), Th

In [17]:
sampled_topic_response = resp_sampled.json()['choices'][0]['message']['content']
print(json.dumps(json.loads(sampled_topic_response), indent=2, ensure_ascii=False))

{
  "topics": [
    {
      "topic_name": "Hotel Reviews in London",
      "topic_description": "Analysis of reviews and complaints about various hotels in London, focusing on key issues and common themes."
    },
    {
      "topic_name": "Positive Aspects",
      "topic_description": "Key positives mentioned in the reviews: good locations, clean rooms, friendly staff, and value for money."
    },
    {
      "topic_name": "Negative Aspects",
      "topic_description": "Common complaints: lack of air conditioning, rude staff, high prices, poor room conditions, and issues with check-in procedures."
    },
    {
      "topic_name": "Hotels Mentioned",
      "topic_description": "Examples of hotels reviewed: The London Farringdon (affordable, good location), The Balmoral (high prices, small rooms), and others with mixed experiences."
    },
    {
      "topic_name": "Overall Consensus",
      "topic_description": "Users generally agree on the need for better amenities and customer servic

In [18]:
import requests

# Structured Output: JSON Schema で出力形式を厳密に強制する
topic_list_schema = {
    "type": "object",
    "properties": {
        "topics": {
            "type": "array",
            "items": {"type": "string"} 
        }
    },
    "required": ["topics"]
}

def extract_topics_from_review(customer_review, topic_list_str):
        delimiter = '####'
        system_message = "You're a helpful assistant. Your task is to analyse hotel reviews."
        user_message = f'''
        Below is a customer review delimited with {delimiter}. 
        Please, identify the main topics mentioned in this comment from the list of topics below.

        Return a list of the relevant topics for the customer review. 

        Output is a JSON list with the following format
        ["<topic1>", "<topic2>", ...]

        If topics are not relevant to the customer review, return an empty list ([]).
        Include only topics from the provided below list.

        List of topics:
        {topic_list_str}

        Customer review:
        {delimiter}
        {customer_review}
        {delimiter}
        '''

        messages =  [  
                {'role':'system', 
                'content': system_message},    
                {'role':'user', 
                'content': f"{user_message}"},  
        ]

        resp = requests.post(
        'http://host.docker.internal:11434/v1/chat/completions',
        json={
                "model": "qwen3:1.7b",
                "messages": messages,
                "response_format": {
                "type": "json_schema",
                "json_schema": {
                        "name": "topics",
                        "schema": topic_list_schema
                }
                },
        }
        )
        # print(resp.status_code)
        return resp.json()

In [19]:
topics = topic_response['topics']
topic_list_str = '\n'.join(map(lambda x: x['topic_name'], topic_response['topics']))
topic_desc_list_str = '\n'.join(map(lambda x: x['topic_name'] + ': ' + x['topic_description'], topic_response['topics']))

In [20]:
resp_review_topics = extract_topics_from_review(df.reviews_transl[0], topic_list_str)
review_topics = json.loads(resp_review_topics['choices'][0]['message']['content'])['topics']
review_topics

['Location', 'Room Facilities', 'Price and Value', 'Specific Issues']

In [21]:
for doc in docs[:5]:
    resp_review_topics1 = extract_topics_from_review(doc, topic_list_str)
    resp_review_topics2 = extract_topics_from_review(doc, topic_desc_list_str)
    review_topics1 = json.loads(resp_review_topics1['choices'][0]['message']['content'])['topics']
    review_topics2 = json.loads(resp_review_topics2['choices'][0]['message']['content'])['topics']
    print(f"Review: {doc}\nAssigned topics (names only): {review_topics1}\nAssigned topics (with descriptions): {review_topics2}\n{'-'*50}")

Review: We stayed three nights over Thanksgiving weekend. While it's a ways out of downtown London, the little town it's in is great--lots of delicious restaurants, inexpensive shopping, close to a fairly major highway. Our two rooms were spare but comfortable and very clean. Bathrooms were fine. Mattresses were sad, saggy and lumpy, but at least not rock hard and the sheets fit and stayed on well. The comforter was just the perfect weight for the weather--we all slept well. There's an in-house restaurant with room service, which we never had a chance to use. The a la carte prices were reasonable, except the buffet breakfast, which was high. It's difficult to find the parking lots, but once you figure out the one way streets and round-abouts, there is safe, adequate parking. We'd go back. For only nine pounds a night for two people, it's a great bargain!
Assigned topics (names only): ['Location', 'Room Facilities', 'Price and Value', 'Specific Issues']
Assigned topics (with description